# Concrete Paving Slab LCA

Cradle-to-grave carbon footprint of one square metre of concrete paving slab, Brightway for the calculation, BAFU for the background.

**This was given on Internet** 

Functional unit: 1 square metre of paving slab, at a typical 5 cm thickness. Concrete's density is about 2400 kg per cubic metre, so that works out to roughly 120 kg per slab, that's the mass everything below is built around.

What's in here:
1. Project setup
2. Import BAFU (ecoSpold1), same debugged procedure as the other notebooks
3. Search BAFU for concrete production and paving-related processes
4. Confirmed picks: transport and disposal, reused from earlier notebooks
5. Build the foreground activity
6. Run the LCA
7. Contribution breakdown by stage
8. Monte Carlo

## 1. Project setup

Its own project again, so it doesn't touch the other three.

In [19]:
import brightway2 as bw

bw.projects.set_current("Concrete Paving Slab LCA")
bw.bw2setup()

print(list(bw.databases))

Biosphere database already present!!! No setup is needed
['biosphere3', 'BAFU', 'Concrete Paving Slab']


## 2. Import BAFU (ecoSpold1)

Same import procedure worked out in the PET bottle notebook: clean the `None` fields before hashing, run bw2io's standard strategies, fix BAFU's older category naming so it lines up with `biosphere3`, then drop what's still unlinked and write the database.

Update the path below to the extracted ecoSpold1 folder.

In [2]:
import bw2io as bi

ECOSPOLD_PATH = "/Users/abhi/ecoSpold files"

importer = bi.SingleOutputEcospold1Importer(ECOSPOLD_PATH, "BAFU")

Extracting XML data from 11947 datasets


/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(mean)),
/opt/anaconda3/envs/ab/lib/python3.11/site-packages/bw2io/extractors/ecospold1.py:320: RuntimeWarning: divide by zero encountered in log
  "loc": np.log(np.abs(

Extracted 11947 datasets in 6.60 seconds


In [3]:
fields_to_clean = ["name", "categories", "unit", "reference product", "location"]

for ds in importer.data:
    for field in fields_to_clean:
        if ds.get(field) is None:
            ds[field] = ""
    for exc in ds.get("exchanges", []):
        for field in fields_to_clean:
            if exc.get(field) is None:
                exc[field] = ""

In [4]:
importer.apply_strategies()
importer.match_database("biosphere3", fields=["name", "categories"])
importer.statistics()

Applying strategy: normalize_units
Applying strategy: assign_only_product_as_production
Applying strategy: clean_integer_codes
Applying strategy: drop_unspecified_subcategories
Applying strategy: normalize_biosphere_categories
Applying strategy: normalize_biosphere_names
Applying strategy: strip_biosphere_exc_locations
Applying strategy: update_ecoinvent_locations
Applying strategy: set_code_by_activity_hash
Applying strategy: link_iterable_by_fields
Applying strategy: link_technosphere_by_activity_hash
Applied 11 strategies in 8.26 seconds
Applying strategy: link_iterable_by_fields
11947 datasets
420063 exchanges
295213 unlinked exchanges
  Type biosphere: 2679 unique unlinked exchanges
  Type technosphere: 24 unique unlinked exchanges


(11947, 420063, 295213)

In [5]:
CATEGORY_FIX = {
    ("emissions to air",): ("air",),
    ("emissions to air", "high. pop."): ("air", "urban air close to ground"),
    ("emissions to air", "low. pop."): ("air", "non-urban air or from high stacks"),
    ("emissions to air", "stratosphere + troposphere"): ("air", "lower stratosphere + upper troposphere"),
    ("emissions to soil",): ("soil",),
    ("emissions to soil", "agricultural"): ("soil", "agricultural"),
    ("emissions to soil", "industrial"): ("soil", "industrial"),
    ("emissions to water",): ("water",),
    ("emissions to water", "ground-"): ("water", "ground-"),
    ("emissions to water", "ocean"): ("water", "ocean"),
    ("emissions to water", "river"): ("water", "surface water"),
    ("resources",): ("natural resource",),
    ("resources", "land"): ("natural resource", "land"),
    ("resources", "in ground"): ("natural resource", "in ground"),
    ("resources", "in air"): ("natural resource", "in air"),
    ("resources", "in water"): ("natural resource", "in water"),
    ("resources", "biotic"): ("natural resource", "biotic"),
}

fixed_count = 0
for ds in importer.data:
    for exc in ds.get("exchanges", []):
        if exc.get("type") == "biosphere":
            cats = tuple(exc.get("categories") or ())
            if cats in CATEGORY_FIX:
                exc["categories"] = CATEGORY_FIX[cats]
                fixed_count += 1

print(f"Rewrote categories on {fixed_count} exchanges")

Rewrote categories on 268970 exchanges


In [6]:
importer.match_database("biosphere3", fields=["name", "categories"])
importer.statistics()

Applying strategy: link_iterable_by_fields
11947 datasets
420063 exchanges
114060 unlinked exchanges
  Type biosphere: 1155 unique unlinked exchanges
  Type technosphere: 24 unique unlinked exchanges


(11947, 420063, 114060)

Check what's still unlinked before dropping it, same as before.

In [7]:
unlinked = list(importer.unlinked)
bio_unlinked = [u for u in unlinked if u.get("type") == "biosphere"]
tech_unlinked = [u for u in unlinked if u.get("type") == "technosphere"]

print(f"{len(bio_unlinked)} unique unlinked biosphere flows")
for u in bio_unlinked[:20]:
    print(u.get("name"), "|", u.get("categories"), "|", u.get("unit"))

print(f"\n{len(tech_unlinked)} unique unlinked technosphere")
for u in tech_unlinked[:24]:
    print(u.get("name"), "|", u.get("location"), "|", u.get("unit"))

1155 unique unlinked biosphere flows
NMVOC, non-methane volatile organic compounds, unspecified origin | ('air', 'non-urban air or from high stacks') | kilogram
Mercury | ('air', 'non-urban air or from high stacks') | kilogram
Occupation, industrial area, built up | ('natural resource', 'land') | square meter-year
Transformation, to industrial area, built up | ('natural resource', 'land') | square meter
Particulates, < 2.5 um | ('air', 'urban air close to ground') | kilogram
Copper | ('air', 'urban air close to ground') | kilogram
Particulates, < 2.5 um | ('air', 'non-urban air or from high stacks') | kilogram
Mercury | ('air', 'urban air close to ground') | kilogram
Selenium | ('air', 'non-urban air or from high stacks') | kilogram
Selenium | ('air', 'urban air close to ground') | kilogram
Nickel | ('air', 'non-urban air or from high stacks') | kilogram
Zinc | ('air', 'non-urban air or from high stacks') | kilogram
Nickel | ('air', 'urban air close to ground') | kilogram
Zinc | ('air'

In [8]:
importer.drop_unlinked(i_am_reckless=True)
importer.statistics()

Applying strategy: drop_unlinked
Applied 1 strategies in 0.20 seconds
11947 datasets
306003 exchanges
0 unlinked exchanges
  


(11947, 306003, 0)

In [9]:
importer.write_database()
print(list(bw.databases))

Title: Writing activities to SQLite3 database:
  Started: 08/06/2026 11:15:26
  Finished: 08/06/2026 11:15:34
  Total time elapsed: 00:00:07
  CPU %: 94.90
  Memory %: 5.12
Created database: BAFU
['biosphere3', 'BAFU']


## 3. Search BAFU for concrete production and paving-related processes

Run each search and look at the real results before picking anything. Concrete has a lot of variants in most inventories (strength class, aggregate type, precast versus in-situ), so expect a long list and pick the one that best matches a standard paving-grade mix.

In [10]:
ei = bw.Database("BAFU")

# Ready-mix or precast concrete production
for r in ei.search("concrete"):
    print(r, "|", r["location"])

'Precast concrete, standard concrete, at plant' (cubic meter, CH, ['construction materials', 'concrete']) | CH
'Precast concrete, high performance concrete, at plant' (cubic meter, CH, ['construction materials', 'concrete']) | CH
'Poor concrete, at plant' (cubic meter, CH, ['construction', 'concrete']) | CH
'Concrete, exacting, at plant' (cubic meter, CH, ['construction', 'concrete']) | CH
'Concrete, normal, at plant' (cubic meter, CH, ['construction', 'concrete']) | CH
'Precast concrete, standard concrete, only common base, at plant' (cubic meter, CH, ['construction materials', 'concrete']) | CH
'Precast concrete, high performance concrete, only common base, at plant' (cubic meter, CH, ['construction materials', 'concrete']) | CH
'Concrete mixing plant' (unit, CH, ['construction', 'concrete\\infrastructure']) | CH
'Lean concrete, unspecific, at plant' (cubic meter, CH, ['construction materials', 'concrete']) | CH
'Concrete, sole plate and foundation, at plant' (cubic meter, CH, ['cons

In [11]:
# Cement is the energy-intensive binder in concrete, worth checking separately
# in case the model needs to be built up from raw materials rather than a
# single ready-mix concrete activity
for r in ei.search("cement"):
    print(r, "|", r["location"])

'Cement plant' (unit, CH, ['construction', 'binders\\infrastructure']) | CH
'Packing, cement' (kilogram, CH, ['Others']) | CH
'Cement, unspecified, at plant' (kilogram, CH, ['construction', 'binders']) | CH
'CEM I cement, at plant' (kilogram, CH, ['construction materials', 'binder']) | CH
'White cement, at plant' (kilogram, CH, ['construction materials', 'plaster']) | CH
'Cement mortar, at plant' (kilogram, CH, ['construction', 'binders']) | CH
'CEM II, B cement, at plant' (kilogram, CH, ['construction materials', 'binder']) | CH
'CEM II, A cement, at plant' (kilogram, CH, ['construction materials', 'binder']) | CH
'Portland calcareous cement, at plant' (kilogram, CH, ['construction', 'binders']) | CH
'CEM III, B cement, at plant' (kilogram, CH, ['construction materials', 'binder']) | CH
'CEM III, A cement, at plant' (kilogram, CH, ['construction materials', 'binder']) | CH
'Packing, fibre cement products' (kilogram, CH, ['Others']) | CH
'CEM II, B-LL cement, at plant' (kilogram, CH, [

In [12]:
# Paving slab specifically, in case BAFU has the finished product
# rather than just the raw concrete mix
for r in ei.search("paving"):
    print(r, "|", r["location"])

print("---")
for r in ei.search("slab"):
    print(r, "|", r["location"])

'Platform, standard' (square meter-year, CH, ['transport systems', 'train\\operations\\infrastructure']) | CH
---
'Dung slab' (square meter, CH, ['agricultural', 'operations\\infrastructure']) | CH
'Polystyrene foam slab, at plant' (kilogram, RER, ['construction', 'insulation']) | RER
'Cork slab, at plant' (kilogram, RER, ['construction', 'insulation']) | RER
'Polystyrene foam slab, 45% recycled, at plant' (kilogram, CH, ['construction', 'insulation']) | CH
'xx Polystyrene foam slab, 100% recycled, at plant' (kilogram, CH, ['material, obsolete', 'construction, obsolete\\insulation, obsolete']) | CH
'Fibre cement corrugated slab, at plant' (kilogram, CH, ['construction', 'coverings']) | CH
'Disposal, cement-fibre slab, 0% water, to municipal incineration' (kilogram, CH, ['waste management', 'municipal incineration']) | CH
'xx Urea formaldehyde foam slab, hard, at plant' (kilogram, CH, ['material, obsolete', 'construction, obsolete\\insulation, obsolete']) | CH
'xx Disposal, building, ce

Fill in the pick below once the searches above have been run and the correct index is known.

In [13]:
# Replace the "..." and [0] index with the confirmed pick, once the search
# results above have been checked
# concrete_mix = ei.search("...")[0]

## 4. Confirmed picks: transport and disposal

Transport reuses the same CH fleet-average lorry pick confirmed across the PET bottle and aluminium can notebooks. For end-of-life, `'Disposal, inert material, 0% water, to sanitary landfill'` already turned up directly in the `sanitary landfill` search results from those earlier notebooks, concrete rubble in Switzerland is typically crushed and landfilled as inert material rather than incinerated, so this is both a confirmed BAFU activity and a genuinely appropriate match, not a proxy.

In [14]:
truck = ei.search("transport lorry")[2]
print(truck)  # 'Transport, freight, lorry, fleet average' (ton kilometer, CH)

landfill = ei.search("sanitary landfill")[3]
print(landfill)  # 'Disposal, inert material, 0% water, to sanitary landfill' (CH)

'Transport, freight, lorry, fleet average' (ton kilometer, CH, ['transport systems', 'road\\Transformation'])
'Disposal, inert material, 0% water, to sanitary landfill' (kilogram, CH, ['landfill', 'sanitary landfill'])


Check both printed lines against the comments before continuing. Index positions can shift slightly between BAFU versions, so confirm rather than assume, same reminder as always.

## 5. Build the foreground activity

Physical parameters first, then the activity itself. Concrete slabs don't really have a meaningful "recycling" end-of-life the way the aluminium can does, crushed concrete gets used as inert fill or landfilled rather than remelted, so there's just one end-of-life pathway here rather than a three-way split.

In [15]:
SLAB_THICKNESS_M = 0.05      # 5 cm, typical for a pedestrian paving slab
CONCRETE_DENSITY_KG_M3 = 2400 # typical density of standard concrete
SLAB_MASS_KG = SLAB_THICKNESS_M * CONCRETE_DENSITY_KG_M3  # per square metre

DIST_KM = 50  # plant to site distance, shorter than the packaging notebooks since
               # concrete is heavy and typically sourced locally

print(f"Slab mass: {SLAB_MASS_KG:.1f} kg per square metre")

Slab mass: 120.0 kg per square metre


In [16]:
DB_NAME = "Concrete Paving Slab"

if DB_NAME in bw.databases:
    del bw.databases[DB_NAME]

exchanges = [
    {"input": (DB_NAME, "concrete_slab_1m2"), "amount": 1, "unit": "square meter", "type": "production"},

    # ---- Material stage ----
    # Uncomment once the pick from section 3 is confirmed:
    # {"input": concrete_mix.key, "amount": SLAB_MASS_KG, "unit": "kilogram", "type": "technosphere"},

    # ---- Distribution ----
    {"input": truck.key, "amount": SLAB_MASS_KG * DIST_KM / 1000, "unit": "ton kilometer", "type": "technosphere"},

    # ---- End of life ----
    {"input": landfill.key, "amount": SLAB_MASS_KG, "unit": "kilogram", "type": "technosphere"},
]

foreground = bw.Database(DB_NAME)
foreground.write({
    (DB_NAME, "concrete_slab_1m2"): {
        "name": "Concrete paving slab, 1 square metre, cradle-to-grave",
        "reference product": "Concrete paving slab, 1 square metre, cradle-to-grave",
        "unit": "square meter",
        "location": "CH",
        "type": "process",
        "exchanges": exchanges,
    }
})

print("Database written:", list(bw.databases))

Title: Writing activities to SQLite3 database:
  Started: 08/06/2026 11:15:54
  Finished: 08/06/2026 11:15:54
  Total time elapsed: 00:00:00
  CPU %: 95.30
  Memory %: 2.96
Database written: ['biosphere3', 'BAFU', 'Concrete Paving Slab']


Note to self: until the material pick is uncommented, this only accounts for transport and disposal, both fairly minor stages for concrete. The score in section 6 will be far too low until the concrete mix itself is filled in and this cell is rerun.

## 6. Run the LCA

Same method as the other notebooks.

In [17]:
slab = bw.get_activity((DB_NAME, "concrete_slab_1m2"))

method = ("IPCC 2013", "climate change", "global warming potential (GWP100)")

lca = bw.LCA({slab: 1}, method)
lca.lci()
lca.lcia()

print(f"GWP 100a: {lca.score:.4f} kg CO2-eq per square metre")

GWP 100a: 2.5350 kg CO2-eq per square metre


## 7. Monte Carlo uncertainty analysis

The single `GWP 100a` score above is one point estimate, it hides how much the result could vary given real uncertainty in every exchange behind it. Monte Carlo reruns the calculation many times, resampling every exchange from its uncertainty distribution each time, and returns a spread instead of one number.

Two things to know first: `bw2calc`'s Monte Carlo solver can raise `ValueError: 'scipy.sparse.linalg.cgs' called with invalid atol='legacy'` on newer `scipy` versions, if that happens, run `conda install "scipy<1.12"` in the `ab` environment and restart the kernel. And Monte Carlo only produces a real spread if the exchanges being sampled actually carry uncertainty data, the foreground exchanges written here don't have any set by default, so they'll stay fixed across iterations unless it's added explicitly.

In [18]:
from bw2calc import MonteCarloLCA

method = ("IPCC 2013", "climate change", "global warming potential (GWP100)")

mc = MonteCarloLCA({slab: 1}, method)
n_iterations = 1000
scores = [next(mc) for _ in range(n_iterations)]

import numpy as np

print(f"Iterations: {n_iterations}")
print(f"Mean:   {np.mean(scores):.4f} kg CO2-eq per square metre")
print(f"Median: {np.median(scores):.4f} kg CO2-eq per square metre")
print(f"Std dev: {np.std(scores):.4f}")
print(f"5th to 95th percentile: {np.percentile(scores, 5):.4f} to {np.percentile(scores, 95):.4f} kg CO2-eq")

Iterations: 1000
Mean:   2.7576 kg CO2-eq per square metre
Median: 2.7061 kg CO2-eq per square metre
Std dev: 0.3478
5th to 95th percentile: 2.2989 to 3.3716 kg CO2-eq
